In [0]:
%sql
create volume if not exists dbxsoumik.bronze.autovol

# **Autoloader query**

In [0]:
## schemaEvolutionMode rescue --> dont add 100 col rather crete dict with 100 ; addnewcolumn 
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation","/Volumes/dbxsoumik/bronze/autovol/destination/checkpoint/")\
  .option("cloudFiles.schemaEvolutionMode", "rescue")\
  .load("dbfs:/Volumes/dbxsoumik/bronze/autovol/raw")

## While adding the new file of same schema only run the write query __

In [0]:
## trigger once true will start and stop the cluster save money
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/dbxsoumik/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/dbxsoumik/bronze/autovol/destination/data/")

In [0]:
spark.read.format("delta")\
.load("/Volumes/dbxsoumik/bronze/autovol/destination/data/").display()

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

#define schema 
resuced_schema = StructType()\
    .add("discount", StringType())\
    .add("payment_method", StringType())

# Parse the stringfield Json column
df_r = df.withColumn("rescued_struct", from_json(col("_rescued_data"),resuced_schema))

# extract individual field
df_n = df_r.withColumn("rescued_discount", col("rescued_struct.discount"))\
           .withColumn("rescued_payment_method", col("rescued_struct.payment_method"))


In [0]:
display(df_n , checkpointLocation = "/Volumes/dbxsoumik/bronze/autovol/destination/checkpoint/")

## **Add new column**

In [0]:
## will create new destoination table and delete src file with new col

dbutils.fs.rm("/Volumes/dbxsoumik/bronze/autovol/destination/checkpoint/",True)
dbutils.fs.rm("/Volumes/dbxsoumik/bronze/autovol/destination/data/",True)
dbutils.fs.rm("/Volumes/dbxsoumik/bronze/autovol/raw/orders_batch3*/",True)


In [0]:

df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation","/Volumes/dbxsoumik/bronze/autovol/destination/checkpoint/")\
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
  .load("dbfs:/Volumes/dbxsoumik/bronze/autovol/raw")

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/dbxsoumik/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .option("Mergeschema",True)\
    .start("/Volumes/dbxsoumik/bronze/autovol/destination/data/")

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/dbxsoumik/bronze/autovol/destination/data/")
display(df)